In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, Conv2D, MaxPooling2D
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

print(f"TensorFlow version: {tf.__version__}")

# Load dataset
try:
    print("Attempting to download CIFAR-10 dataset...")
    (x_train, y_train), (x_test, y_test) = cifar10.load_data()
    print("Dataset loaded successfully!")
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Please try manual download or check your internet connection.")
    exit()

# Class names for CIFAR-10
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# Display first 9 training images
print("\nDisplaying sample training images...")
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')
plt.suptitle('Sample Training Images', fontsize=16)
plt.tight_layout()
plt.show()

# Preprocess data
num_classes = 10
y_train_original = y_train.copy()
y_test_original = y_test.copy()

y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255

print('\nx_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

# Build model
print("\nBuilding model...")
model = Sequential()
model.add(Conv2D(32, (3, 3), input_shape=(32, 32, 3), activation='relu', padding='same'))
model.add(Dropout(0.2))
model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(Dropout(0.2))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(Dropout(0.2))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())
model.add(Dropout(0.2))
model.add(Dense(1024, activation='relu', kernel_constraint=max_norm(3)))
model.add(Dropout(0.2))
model.add(Dense(512, activation='relu', kernel_constraint=max_norm(3)))
model.add(Dropout(0.2))
model.add(Dense(num_classes, activation='softmax'))

print(model.summary())

# Compile model
opt = SGD(learning_rate=0.01, momentum=0.9, nesterov=False)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

# Train model
print("\nStarting model training...")
epochs = 10
batch_size = 32
history = model.fit(x_train, y_train,
                    batch_size=batch_size,
                    epochs=epochs,
                    validation_data=(x_test, y_test),
                    shuffle=True,
                    verbose=1)

# Evaluate model
print("\nEvaluating model...")
scores = model.evaluate(x_test, y_test, verbose=1)
print('Test loss:', scores[0])
print('Test accuracy:', scores[1])

# Save model
print("\nSaving model...")
model.save('classifier.h5')
print("Model saved as 'classifier.h5'")

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("MAKING PREDICTIONS ON TEST IMAGES")
print("="*70)

# Make predictions on first 10 test images
num_predictions = 10
predictions = model.predict(x_test[:num_predictions])

# Display predictions
plt.figure(figsize=(15, 8))
for i in range(num_predictions):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i])
    
    predicted_class = predictions[i].argmax()
    true_class = y_test_original[i][0]
    confidence = predictions[i][predicted_class] * 100
    
    color = 'green' if predicted_class == true_class else 'red'
    
    plt.title(f"Pred: {class_names[predicted_class]}\n"
              f"True: {class_names[true_class]}\n"
              f"Conf: {confidence:.1f}%",
              color=color, fontsize=9)
    plt.axis('off')

plt.suptitle('Predictions on Test Images (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.show()

# Detailed probability breakdown for first 5 images
print("\nDETAILED PROBABILITY BREAKDOWN:")
print("="*70)
for img_idx in range(5):
    print(f"\n{'IMAGE #{img_idx + 1}'.center(70, '-')}")
    print(f"True Label: {class_names[y_test_original[img_idx][0]]}")
    print(f"\nPrediction Probabilities:")
    print("-" * 70)
    
    sorted_indices = np.argsort(predictions[img_idx])[::-1]
    
    for rank, idx in enumerate(sorted_indices, 1):
        prob = predictions[img_idx][idx] * 100
        bar = '█' * int(prob / 2)
        marker = " ← PREDICTED" if idx == predictions[img_idx].argmax() else ""
        print(f"{rank:2d}. {class_names[idx]:12s}: {prob:6.2f}% {bar}{marker}")

correct_predictions = sum([1 for i in range(num_predictions) 
                          if predictions[i].argmax() == y_test_original[i][0]])
print("\n" + "="*70)
print(f"SUMMARY: {correct_predictions}/{num_predictions} predictions correct "
      f"({correct_predictions/num_predictions*100:.1f}%)")
print("="*70)

# =====================================================================
# NEW SECTION: PREDICT ON CUSTOM IMAGE
# =====================================================================
print("\n" + "="*70)
print("PREDICTING ON YOUR CUSTOM IMAGE")
print("="*70)

def preprocess_custom_image(image_path):
    """
    Load and preprocess a custom image for prediction.
    The image will be resized to 32x32 pixels to match CIFAR-10 format.
    """
    try:
        # Load image
        img = Image.open(image_path)
        
        # Convert to RGB if necessary
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        # Resize to 32x32 (CIFAR-10 size)
        img = img.resize((32, 32))
        
        # Convert to numpy array
        img_array = np.array(img)
        
        # Normalize pixel values (0-255 to 0-1)
        img_array = img_array.astype('float32') / 255.0
        
        # Add batch dimension
        img_array = np.expand_dims(img_array, axis=0)
        
        return img_array, img
    except Exception as e:
        print(f"Error loading image: {e}")
        return None, None

# PATH TO YOUR IMAGE - CHANGE THIS TO YOUR IMAGE PATH
custom_image_path = 'your_image.jpg'  # ← PUT YOUR IMAGE PATH HERE

print(f"\nAttempting to load image from: {custom_image_path}")
processed_img, original_img = preprocess_custom_image(custom_image_path)

if processed_img is not None:
    # Make prediction
    print("Making prediction on custom image...")
    custom_prediction = model.predict(processed_img, verbose=0)
    
    # Get predicted class and confidence
    predicted_class_idx = custom_prediction[0].argmax()
    predicted_class_name = class_names[predicted_class_idx]
    confidence = custom_prediction[0][predicted_class_idx] * 100
    
    # Display the image with prediction
    plt.figure(figsize=(12, 5))
    
    # Show original resized image
    plt.subplot(1, 2, 1)
    plt.imshow(original_img)
    plt.title(f"Your Custom Image\n(Resized to 32x32)", fontsize=12)
    plt.axis('off')
    
    # Show prediction probabilities as bar chart
    plt.subplot(1, 2, 2)
    sorted_indices = np.argsort(custom_prediction[0])[::-1]
    top_5_indices = sorted_indices[:5]
    top_5_probs = [custom_prediction[0][i] * 100 for i in top_5_indices]
    top_5_names = [class_names[i] for i in top_5_indices]
    
    colors = ['green' if i == 0 else 'skyblue' for i in range(5)]
    plt.barh(range(5), top_5_probs, color=colors)
    plt.yticks(range(5), top_5_names)
    plt.xlabel('Probability (%)')
    plt.title(f'Top 5 Predictions\nPredicted: {predicted_class_name} ({confidence:.1f}%)', 
              fontsize=12)
    plt.xlim(0, 100)
    
    # Add percentage labels on bars
    for i, prob in enumerate(top_5_probs):
        plt.text(prob + 1, i, f'{prob:.1f}%', va='center')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed results
    print("\n" + "-"*70)
    print(f"PREDICTION RESULT FOR CUSTOM IMAGE")
    print("-"*70)
    print(f"Predicted Class: {predicted_class_name}")
    print(f"Confidence: {confidence:.2f}%")
    print("\nAll Class Probabilities:")
    print("-"*70)
    
    for rank, idx in enumerate(sorted_indices, 1):
        prob = custom_prediction[0][idx] * 100
        bar = '█' * int(prob / 2)
        marker = " ← PREDICTED" if idx == predicted_class_idx else ""
        print(f"{rank:2d}. {class_names[idx]:12s}: {prob:6.2f}% {bar}{marker}")
    
    print("="*70)
else:
    print(f"\nCould not load image from '{custom_image_path}'")
    print("Please make sure:")
    print("1. The image file exists at the specified path")
    print("2. The file is a valid image format (jpg, png, etc.)")
    print("3. Update the 'custom_image_path' variable with your image path")

print("\nTraining complete! Model ready for use.")
print("\nNOTE: CIFAR-10 images are very small (32x32 pixels), so the model")
print("works best with simple images. High-resolution photos will be resized")
print("and may lose detail, affecting prediction accuracy.")


ModuleNotFoundError: No module named 'tensorflow'